# ER Patient Triage Optimization

| Field | Value |
|---|---|
| **Author** | Zachary Failor |
| **Course** | ADS 5834 — Data Science in Practice |
| **Notebook** | data_preparation |
| **Created Date** | 2026-11-16 |
| **Last Updated Date** | 2026-11-19 |
| **Last Updated By** | Adam Bergen |
| **Technical Owner Email** | zachf@vt.edu |
| **Business Owner Email** | zachf@vt.edu |

## Project Description
Predict patient severity and optimize assignments to beds, physicians, and time slots to improve care effectiveness while consiering the hospital resource limits.

## Notebook Description
Select, clean, construct, integrate, and format the patient and hospital resource data to create the final datasets needed for severity prediction and ER scheduling.

## 3.1 Select Data

| Data | Decision | Reason |
|---|---|---|
| Historical patients (4,000 rows) | Include | It has the severity scores needed to train the model. |
| Scheduling patients (177 rows) | Include | These are the patients that need severity predictions and scheduling |
| Beds and physicians | Include | Shows resources and how many patients they can handle. |
| Care quality | Include | Scores each bed-type and physician-type combination. |
| Time configuration | Include | Defines 24 slots, half-hour each, and starts at 6 PM. |
| Existing optimization metrics | Exclude | Reports 108 scheduled patients. We need to verify these results before using them for comparison. |
| patient_id | Identifier | Tracks patients and not used for predictions. |
| severity_score | Target | The score from 1–5 that the model will predict. |
| Age, sex, and chief complaint | Predictors | Available in patient tables and required. |
| heart_rate, bp_systolic, bp_diastolic, temperature, respiratory_rate, o2_sat, pain_score | Predictors | Vital signs and pain are predictors and are related to severity |
| Diabetes, hypertension, and heart disease | Predictors | Describe the patient's recorded existing conditions. |
| Ambulance arrival | Predictor | Known when the patient arrives and is in both tables. |
| arrival_time | Scheduling | Prevents patients from being scheduled before they arrive. |

Constraints: The data is small size, so there are no volume limits. Every column is something that can be enconded like an integer, double, or string.

In [0]:
import pandas as pd
import numpy as np

SOURCE = "workspace.healthcare_triage"
DESTINATION = "workspace.dm_final_project"

table_names = [
    "er_patient_history",
    "er_scheduling_input",
    "er_resources_beds",
    "er_resources_physicians",
    "er_resources_care_quality",
    "er_resources_time_config"
]

raw = {
    name: spark.read.table(f"{SOURCE}.{name}").toPandas()
    for name in table_names
}

history = raw["er_patient_history"].copy()
scheduling = raw["er_scheduling_input"].copy()
beds = raw["er_resources_beds"].copy()
physicians = raw["er_resources_physicians"].copy()
care_quality = raw["er_resources_care_quality"].copy()
time_config = raw["er_resources_time_config"].copy()

categorical_features = ["sex", "chief_complaint"]

numeric_features = [
    "age", "heart_rate", "bp_systolic", "bp_diastolic",
    "temperature", "respiratory_rate", "o2_sat", "pain_score",
    "diabetes", "hypertension", "heart_disease", "ambulance_arrival"
]

feature_columns = numeric_features + categorical_features

history = history[
    ["patient_id"] + feature_columns + ["severity_score"]
].copy()

scheduling = scheduling[
    ["patient_id", "arrival_time"] + feature_columns
].copy()

display(pd.DataFrame({
    "table": table_names,
    "rows": [len(raw[name]) for name in table_names]
}))

table,rows
er_patient_history,4000
er_scheduling_input,177
er_resources_beds,25
er_resources_physicians,6
er_resources_care_quality,6
er_resources_time_config,4


## 3.2 Clean Data

Data Understanding found no missing values or duplicate records. There is no need to fill in values or remove duplicates from this result.

One historical patient has systolic blood pressure below diastolic blood pressure. We will flag it for review and leave it out of the data until it can be verified. This will leave 3,999 historical records.

Arrival times will be converted from strings to timestamps. Also, we will remove extra spaces from category values, make capitalization consistent, and rename the physicians table's type column to physician_type.

Unusual vital signs will stay unless there is evidence that they are incorrect. Removing them just because they are extreme could remove crucial examples of high-severity patients.

These changes will be made to copies of the data. The original warehouse tables will remain unchanged.

In [0]:
# Standardize category formatting.
for frame in [history, scheduling]:
    for column in categorical_features:
        frame[column] = frame[column].str.strip().str.lower()

physicians = physicians.rename(columns={"type": "physician_type"})

for frame, columns in [
    (beds, ["bed_type"]),
    (physicians, ["physician_type"]),
    (care_quality, ["bed_type", "physician_type"])
]:
    for column in columns:
        frame[column] = frame[column].str.strip().str.lower()

# Confirm that missing values or duplicate keys have not appeared.
key_checks = [
    ("history", history, ["patient_id"]),
    ("scheduling", scheduling, ["patient_id"]),
    ("beds", beds, ["bed_id"]),
    ("physicians", physicians, ["physician_id"]),
    ("care_quality", care_quality, ["bed_type", "physician_type"]),
    ("time_config", time_config, ["parameter"])
]

for name, frame, keys in key_checks:
    assert not frame.isna().any().any(), f"Missing values in {name}"
    assert not frame.duplicated(keys).any(), f"Duplicate keys in {name}"

    text_columns = frame.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        assert not frame[column].str.strip().eq("").any(), (
            f"Blank values in {name}.{column}"
        )

# Separate the historical record with inconsistent blood pressure.
invalid_bp = history["bp_systolic"] < history["bp_diastolic"]

history_review = history.loc[invalid_bp].copy()
history_review["review_reason"] = "Systolic pressure below diastolic pressure"

history_clean = history.loc[~invalid_bp].copy()
scheduling_clean = scheduling.copy()

# Convert arrival strings using the format verified previously.
scheduling_clean["arrival_time"] = pd.to_datetime(
    scheduling_clean["arrival_time"],
    format="%m/%d/%Y %H:%M",
    errors="raise"
)

# Repeat the basic patient-value checks before preparing model inputs.
for name, frame in [
    ("history", history_clean),
    ("scheduling", scheduling_clean)
]:
    assert frame["age"].between(0, 120).all(), name
    assert frame["pain_score"].between(0, 10).all(), name
    assert frame["o2_sat"].between(0, 100).all(), name

    for column in [
        "heart_rate", "respiratory_rate", "bp_systolic", "bp_diastolic"
    ]:
        assert frame[column].gt(0).all(), f"{name}: {column}"

    assert frame["bp_systolic"].ge(frame["bp_diastolic"]).all(), name

    for column in [
        "diabetes", "hypertension", "heart_disease", "ambulance_arrival"
    ]:
        assert frame[column].isin([0, 1]).all(), f"{name}: {column}"

assert history_clean["severity_score"].isin([1, 2, 3, 4, 5]).all()

print("Historical records retained:", len(history_clean))
print("Historical records requiring review:", len(history_review))
print("Scheduling patients retained:", len(scheduling_clean))

display(history_review)

Historical records retained: 3999
Historical records requiring review: 1
Scheduling patients retained: 177


patient_id,age,heart_rate,bp_systolic,bp_diastolic,temperature,respiratory_rate,o2_sat,pain_score,diabetes,hypertension,heart_disease,ambulance_arrival,sex,chief_complaint,severity_score,review_reason
TRAIN_3274,44,102,60,66,100.2,22,89,3,0,1,0,0,f,neurological,4,Systolic pressure below diastolic pressure


## 3.3 Construct Data

##### Derived attributes

| New Variable | Table | Calculation | Why We Are Creating It |
|---|---|---|---|
| pulse_pressure | Patients | bp_systolic − bp_diastolic | Gives the difference between the two blood pressures. |
| mean_arterial_pressure | Patients | (bp_systolic + 2 × bp_diastolic) ÷ 3 | Average pressure pushing blood to the organs. Values below 65 suggest poor perfusion, which neither reading shows alone. |
| shock_index | Patients | heart_rate ÷ bp_systolic | Listed in the design document's vitals features. Combines heart rate, the strongest predictor in Data Understanding (r = 0.76), with systolic pressure. Values of 0.9 or higher are a common warning sign. |
| abnormal_vital_count | Patients | Count of: heart_rate > 100 or < 60; bp_systolic < 90; respiratory_rate > 20 or < 12; o2_sat < 92; temperature ≥ 100.4 or < 96.8 °F | Listed in the design document's vitals features. Summarizes how many vital signs are outside normal limits. |
| comorbidity_count | Patients | diabetes + hypertension + heart_disease | Counts how many of these three conditions a patient has. Having more of these conditions helps predict higher patient severity. |
| high_risk_complaint | Patients | 1 if chief_complaint is chest_pain, neurological, or respiratory; otherwise 0 | The design document flags chest pain, stroke symptoms, and shortness of breath as high-risk. In Data Understanding, about 64% of neurological and 59% of chest-pain patients had severity 4–5. |
| arrival_minutes | Scheduling | (arrival_time − window_start) in minutes | Shows how far into the scheduling window each patient arrives. |
| first_eligible_slot | Scheduling | Round arrival_minutes ÷ slot_minutes up to the next integer | Finds the first slot starting at or after arrival. Slots are numbered from 0. |
| no_eligible_slot | Scheduling | first_eligible_slot ≥ number_of_slots | Flags patients with no valid slot in the current window. |
| min_wait_minutes | Scheduling | (first_eligible_slot × slot_minutes) − arrival_minutes; blank when no_eligible_slot | The wait that cannot be avoided because treatment starts at slot boundaries. Needed to compare the optimizer's wait times with the existing metrics. |
| eligible_slot_count | Scheduling | number_of_slots − first_eligible_slot, with a minimum of 0 | How many slots are still available to a patient. Patients with fewer options can be prioritized when capacity is tight. |
| high_severity | Patients | 1 if severity_score ≥ 4; otherwise 0 | Identifies high-severity historical patients for evaluation, not as a predictor. |

The vital-sign limits are fixed clinical cutoffs chosen before modeling. They are not tuned on the data, so they do not leak information from the test set. The respiratory-rate cutoff flags many patients because the average rate is about 20.7; the model can still weigh it against the other vitals.

mean_arterial_pressure, shock_index, abnormal_vital_count, and high_risk_complaint are added to the severity predictors. high_severity and the scheduling variables are not predictors. The historical table has no arrival times, so time-based variables can only be used for scheduling.

##### Generated Records

We will create 24 time-slot records from the time configuration table:

| Field | Calculation |
|---|---|
| slot_id | Numbers 0–23 |
| slot_start | window_start + (slot_id × slot_minutes) |
| slot_end | slot_start + slot_minutes |

The time calculations use minutes.

The final slot starts at 5:30 AM, but some patients arrive later. If treatment starts at slot boundaries, those patients have no valid slot. We will keep and flag them rather than round their arrival times backward. The scheduling window or timing rule needs clarification before every patient can be assigned.

In [ ]:
HIGH_RISK_COMPLAINTS = {"chest_pain", "neurological", "respiratory"}

# Fixed clinical cutoffs; temperature is in Fahrenheit.
ABNORMAL_VITAL_RULES = {
    "heart_rate": lambda s: (s > 100) | (s < 60),
    "bp_systolic": lambda s: s < 90,
    "respiratory_rate": lambda s: (s > 20) | (s < 12),
    "o2_sat": lambda s: s < 92,
    "temperature": lambda s: (s >= 100.4) | (s < 96.8)
}

for frame in [history_clean, scheduling_clean]:
    frame["pulse_pressure"] = (
        frame["bp_systolic"] - frame["bp_diastolic"]
    )

    frame["mean_arterial_pressure"] = (
        frame["bp_systolic"] + 2 * frame["bp_diastolic"]
    ) / 3

    frame["shock_index"] = frame["heart_rate"] / frame["bp_systolic"]

    frame["abnormal_vital_count"] = sum(
        rule(frame[column]).astype(int)
        for column, rule in ABNORMAL_VITAL_RULES.items()
    )

    frame["comorbidity_count"] = frame[
        ["diabetes", "hypertension", "heart_disease"]
    ].sum(axis=1).astype(int)

    frame["high_risk_complaint"] = (
        frame["chief_complaint"].isin(HIGH_RISK_COMPLAINTS).astype(int)
    )

derived_patient_features = [
    "pulse_pressure", "mean_arterial_pressure", "shock_index",
    "abnormal_vital_count", "comorbidity_count", "high_risk_complaint"
]

for column in derived_patient_features:
    if column not in numeric_features:
        numeric_features.append(column)

# Basic checks on the new values.
for name, frame in [("history", history_clean), ("scheduling", scheduling_clean)]:
    assert frame["shock_index"].gt(0).all(), name
    assert frame["mean_arterial_pressure"].between(
        frame["bp_diastolic"], frame["bp_systolic"]
    ).all(), name
    assert frame["abnormal_vital_count"].between(
        0, len(ABNORMAL_VITAL_RULES)
    ).all(), name

history_clean["high_severity"] = (
    history_clean["severity_score"] >= 4
).astype(int)

display(
    history_clean[
        ["patient_id", "chief_complaint"]
        + derived_patient_features
        + ["severity_score", "high_severity"]
    ].head(10)
)

# Average severity by new variable, as a quick sanity check.
display(
    history_clean.groupby("abnormal_vital_count")["severity_score"]
    .agg(["count", "mean"]).round(2)
)
display(
    history_clean.groupby("high_risk_complaint")["severity_score"]
    .agg(["count", "mean"]).round(2)
)

In [ ]:
config = time_config.set_index("parameter")["value"].astype(int).to_dict()

number_of_slots = config["time_slots"]
slot_minutes = config["slot_duration_minutes"]
start_hour = config["start_hour"]
total_hours = config["total_hours"]

assert number_of_slots > 0 and slot_minutes > 0
assert number_of_slots * slot_minutes == total_hours * 60

# Scenario date verified in Data Understanding.
window_start = (
    pd.Timestamp("2024-03-15") + pd.Timedelta(hours=start_hour)
)
window_end = window_start + pd.Timedelta(hours=total_hours)

assert scheduling_clean["arrival_time"].ge(window_start).all()
assert scheduling_clean["arrival_time"].lt(window_end).all()

time_slots = pd.DataFrame({
    "slot_id": np.arange(number_of_slots, dtype=int),
    "slot_start": pd.date_range(
        start=window_start,
        periods=number_of_slots,
        freq=pd.Timedelta(minutes=slot_minutes)
    )
})

time_slots["slot_end"] = (
    time_slots["slot_start"] + pd.Timedelta(minutes=slot_minutes)
)

scheduling_clean["arrival_minutes"] = (
    scheduling_clean["arrival_time"] - window_start
).dt.total_seconds() / 60

scheduling_clean["first_eligible_slot"] = np.ceil(
    scheduling_clean["arrival_minutes"] / slot_minutes
).astype(int)

scheduling_clean["no_eligible_slot"] = (
    scheduling_clean["first_eligible_slot"] >= number_of_slots
)

# Unavoidable wait until the first slot boundary; blank if no slot exists.
scheduling_clean["min_wait_minutes"] = (
    scheduling_clean["first_eligible_slot"] * slot_minutes
    - scheduling_clean["arrival_minutes"]
).where(~scheduling_clean["no_eligible_slot"])

scheduling_clean["eligible_slot_count"] = (
    number_of_slots - scheduling_clean["first_eligible_slot"]
).clip(lower=0).astype(int)

eligible = ~scheduling_clean["no_eligible_slot"]
assert scheduling_clean.loc[eligible, "min_wait_minutes"].between(
    0, slot_minutes, inclusive="left"
).all()
assert scheduling_clean.loc[
    ~eligible, "eligible_slot_count"
].eq(0).all()

display(time_slots)

print(
    "Patients with no eligible slot:",
    int(scheduling_clean["no_eligible_slot"].sum())
)

print(
    "Average unavoidable wait (minutes):",
    round(scheduling_clean["min_wait_minutes"].mean(), 1)
)

display(
    scheduling_clean.loc[
        scheduling_clean["no_eligible_slot"],
        ["patient_id", "arrival_time", "first_eligible_slot",
         "eligible_slot_count"]
    ]
)

## 3.4 Integrate Data

We will combine the beds and physicians tables to create the possible bed–physician pairs. The care quality multiplier will then be added by matching both bed_type and physician_type.

Each row will contain the resource IDs, types, capacities, and care quality multiplier. These are possible choices for the optimizer, not actual assignments. Every pair should have just one matching care quality value.

Historical and scheduling patients will be separate because they have different purposes. After the  model is trained, the predictions will be added to scheduling patients by using patient_id.

Also, we will count arrivals in each 30-minute interval and compare them with physician capacity. This builds on the uneven arrival patterns found in Data Understanding and it tells when new arrivals pass the number physicians can handle. These arrival counts describe demand during each interval. Patients still can't begin treatment before they arrive.

The arrival counts alone do not include patients still waiting from earlier intervals, so we will also add a **backlog** column. It carries unserved patients forward: backlog = max(0, previous backlog + arrivals − physician capacity). This is an estimate of demand pressure, not a schedule, and it assumes every physician is available in every slot.

The physicians can handle 8 patients per slot, giving an upper limit of 192 assignments across 24 slots. That does not guarantee a complete schedule because arrival times and other constraints affect assignments.

In [0]:
assert beds["capacity"].gt(0).all()
assert physicians["max_patients_per_slot"].gt(0).all()
assert care_quality["care_quality_multiplier"].between(0, 1).all()

resource_pairs = beds.merge(physicians, how="cross")

resource_pairs = resource_pairs.merge(
    care_quality,
    on=["bed_type", "physician_type"],
    how="left",
    validate="many_to_one"
)

assert len(resource_pairs) == len(beds) * len(physicians)
assert not resource_pairs.duplicated(["bed_id", "physician_id"]).any()
assert resource_pairs["care_quality_multiplier"].notna().all()

bed_capacity_per_slot = int(beds["capacity"].sum())
physician_capacity_per_slot = int(
    physicians["max_patients_per_slot"].sum()
)

capacity_upper_bound = (
    min(bed_capacity_per_slot, physician_capacity_per_slot)
    * number_of_slots
)

display(resource_pairs.head(10))

display(pd.DataFrame({
    "measure": [
        "Bed capacity per slot",
        "Physician capacity per slot",
        "Assignment capacity upper bound",
        "Patients needing scheduling"
    ],
    "value": [
        bed_capacity_per_slot,
        physician_capacity_per_slot,
        capacity_upper_bound,
        len(scheduling_clean)
    ]
}))

bed_id,bed_type,capacity,physician_id,physician_type,max_patients_per_slot,care_quality_multiplier
ICU_01,icu,1,ATT_01,attending,2,1.0
ICU_01,icu,1,ATT_02,attending,2,1.0
ICU_01,icu,1,RES_01,resident,1,0.8
ICU_01,icu,1,RES_02,resident,1,0.8
ICU_01,icu,1,RES_03,resident,1,0.8
ICU_01,icu,1,RES_04,resident,1,0.8
ICU_02,icu,1,ATT_01,attending,2,1.0
ICU_02,icu,1,ATT_02,attending,2,1.0
ICU_02,icu,1,RES_01,resident,1,0.8
ICU_02,icu,1,RES_02,resident,1,0.8


measure,value
Bed capacity per slot,25
Physician capacity per slot,8
Assignment capacity upper bound,192
Patients needing scheduling,177


In [ ]:
# Floor is used only to group arrivals into reporting intervals.
# It does not change first_eligible_slot or the scheduling rule.
import matplotlib.pyplot as plt
arrival_intervals = np.floor(
    scheduling_clean["arrival_minutes"] / slot_minutes
).astype(int)

arrival_counts = (
    arrival_intervals.value_counts()
    .rename_axis("slot_id")
    .reset_index(name="arrivals")
)

slot_summary = time_slots.merge(
    arrival_counts,
    on="slot_id",
    how="left",
    validate="one_to_one"
)

slot_summary["arrivals"] = (
    slot_summary["arrivals"].fillna(0).astype(int)
)
slot_summary["physician_capacity"] = physician_capacity_per_slot

# Patients left waiting at the end of each interval.
backlog = []
waiting = 0
for arrivals in slot_summary["arrivals"]:
    waiting = max(0, waiting + arrivals - physician_capacity_per_slot)
    backlog.append(waiting)
slot_summary["backlog"] = backlog

assert slot_summary["arrivals"].sum() == len(scheduling_clean)

labels = slot_summary["slot_start"].dt.strftime("%I:%M %p")

fig, ax = plt.subplots(figsize=(11, 4))

ax.bar(labels, slot_summary["arrivals"], label="Arrivals")
ax.axhline(
    physician_capacity_per_slot,
    color="red",
    linestyle="--",
    label=f"Physician capacity per slot ({physician_capacity_per_slot})"
)

ax.plot(
    labels, slot_summary["backlog"],
    color="black", marker="o", label="Backlog (patients still waiting)"
)

ax.set_xlabel("Arrival interval start")
ax.set_ylabel("Number of patients")
ax.set_title("Patient Arrivals Compared with Physician Capacity")
ax.legend()

plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

print("Peak backlog:", int(slot_summary["backlog"].max()))
display(slot_summary)

## 3.5 Format Data

The numeric predictors will be stored consistently, and severity_score will be an integer. Patient IDs will be kept for tracking but excluded from the predictors.

We will split the cleaned historical data into 80% training and 20% testing data. A fixed random seed will make the split repeatable. Since severity classes are uneven, we will stratify by severity_score to keep their proportions similar in both sets.

Sex and chief complaint will be one-hot encoded so the model can use them. The encoder will learn categories from the training data only, then apply the same columns to the testing and scheduling data. Unseen categories will be handled without stopping prediction.

The prepared predictors can be used with a tree-based classifier without scaling. If another model needs scaling, the scaler will be fitted using only training data.

The scheduling records will be sorted by arrival time and patient ID for review. Arrival fields and high_severity will not be included in the severity predictors.

All prepared tables will be saved in workspace.dm_final_project.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

for frame in [history_clean, scheduling_clean]:
    frame[numeric_features] = frame[numeric_features].astype("float64")

history_clean["severity_score"] = (
    history_clean["severity_score"].astype("int64")
)

train_rows, test_rows = train_test_split(
    history_clean,
    test_size=0.20,
    random_state=42,
    stratify=history_clean["severity_score"]
)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
encoder.fit(train_rows[categorical_features])

encoded_columns = encoder.get_feature_names_out(
    categorical_features
).tolist()

model_features = numeric_features + encoded_columns

def format_model_inputs(frame, include_target=False):
    numeric = frame[numeric_features].reset_index(drop=True)

    encoded = pd.DataFrame(
        encoder.transform(frame[categorical_features]),
        columns=encoded_columns
    )

    result = pd.concat([numeric, encoded], axis=1)
    result.insert(0, "patient_id", frame["patient_id"].to_numpy())

    if include_target:
        result["severity_score"] = frame["severity_score"].to_numpy()

    return result

train_prepared = format_model_inputs(train_rows, include_target=True)
test_prepared = format_model_inputs(test_rows, include_target=True)

scheduling_clean = scheduling_clean.sort_values(
    ["arrival_time", "patient_id"]
).reset_index(drop=True)

scheduling_features = format_model_inputs(scheduling_clean)

encoding_map = pd.DataFrame([
    {
        "source_column": column,
        "category": str(category),
        "encoded_column": f"{column}_{category}"
    }
    for column, categories in zip(
        categorical_features, encoder.categories_
    )
    for category in categories
])

In [ ]:
assert set(train_prepared["patient_id"]).isdisjoint(
    set(test_prepared["patient_id"])
)

assert len(train_prepared) + len(test_prepared) == len(history_clean)
assert len(scheduling_features) == len(scheduling)

for frame in [train_prepared, test_prepared, scheduling_features]:
    assert not frame.isna().any().any()
    assert np.isfinite(frame[model_features].to_numpy(dtype=float)).all()

assert train_prepared[model_features].columns.equals(
    test_prepared[model_features].columns
)
assert train_prepared[model_features].columns.equals(
    scheduling_features[model_features].columns
)

display(pd.DataFrame({
    "dataset": ["Training", "Testing", "Scheduling predictors"],
    "rows": [
        len(train_prepared),
        len(test_prepared),
        len(scheduling_features)
    ],
    "predictor_columns": [len(model_features)] * 3
}))

# Confirm similar severity proportions after the split.
display(
    pd.DataFrame({
        "training_percent": (
            train_rows["severity_score"].value_counts(normalize=True) * 100
        ),
        "testing_percent": (
            test_rows["severity_score"].value_counts(normalize=True) * 100
        )
    }).sort_index().round(1)
)

display(encoding_map)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DESTINATION}")

prepared_tables = {
    "er_train_prepared": train_prepared,
    "er_test_prepared": test_prepared,
    "er_scheduling_features": scheduling_features,
    "er_scheduling_prepared": scheduling_clean,
    "er_resource_pairs_prepared": resource_pairs,
    "er_time_slots_prepared": time_slots,
    "er_slot_demand_summary": slot_summary,
    "er_category_encoding": encoding_map
}

if not history_review.empty:
    prepared_tables["er_history_review"] = history_review

saved_results = []

for name, frame in prepared_tables.items():
    full_name = f"{DESTINATION}.{name}"

    (
        spark.createDataFrame(frame)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

    saved_count = spark.read.table(full_name).count()
    assert saved_count == len(frame), f"Row count mismatch: {full_name}"

    saved_results.append({
        "table": full_name,
        "rows": saved_count,
        "columns": len(frame.columns)
    })

display(pd.DataFrame(saved_results))